In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
import panel as pn
from tqdm import tqdm

from scipy import stats
from pathlib import Path
from pprint import pprint
from holoviews import opts
from bokeh.io import output_notebook


font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=600, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Scatter(width=600, height=400, size=8, tools=['hover'], fontsize=font_dict),
    hv.opts.Histogram(width=600, height=400, fontsize=font_dict),
    hv.opts.Bars(width=600, height=400, fontsize=font_dict),
)

output_notebook()
hv.extension('bokeh')


Loading BokehJS ...

In [2]:
monkey = 'fiona' # 'yasmin'  or 'fiona' 

# Load the pickle file
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data'
# pickle_file = save_path / f'unified_{monkey}_cell_trial_data.pkl'
pickle_file = save_path / f'alt_saccade_params_{monkey}_cell_trial_data.pkl'
cell_df = pd.read_pickle(pickle_file)
print(f"Unified DataFrame loaded from: {pickle_file}")
print(f"DataFrame shape: {cell_df.shape}")
print(cell_df.info())
cell_df.head()

Unified DataFrame loaded from: /home/barak/Projects/population-analysis/data/unified_cell_trial_data/alt_saccade_params_fiona_cell_trial_data.pkl
DataFrame shape: (3185424, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3185424 entries, 0 to 3185423
Data columns (total 27 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   cell_ID                 int64  
 1   cell_type               object 
 2   maestro_ID              int64  
 3   problem                 object 
 4   grade                   int64  
 5   filename                object 
 6   trial_name              object 
 7   reaction_time           float64
 8   go_cue                  int64  
 9   stop_cue                float64
 10  trial_failed            bool   
 11  ssd_len                 int64  
 12  ssd_number              float64
 13  type                    object 
 14  first_relevant_saccade  object 
 15  segs_durations          object 
 16  segs_times              object 
 17  tr

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9026,msn,1,waveform looks different,10,fi210713a.0529,STOP_R_SSD2,NaN,921,1005.0,...,1605,0.0,"[[248, 296], [569, 586]]",None,0,[],fi210713,a,529,fi210713a
1,9025,msn,1,NaN,10,fi210713a.0169,STOP_R_SSD4,236.0,921,1125.0,...,1726,0.0,"[[36, 63], [364, 397], [1157, 1190]]",None,0,[],fi210713,a,169,fi210713a
2,9026,msn,1,waveform looks different,10,fi210713a.0457,STOP_L_SSD1,NaN,1093,1117.0,...,1717,0.0,"[[133, 165], [323, 352], [444, 526], [536, 547]]","[338, 444, 447, 449]",180,[],fi210713,a,457,fi210713a
3,9026,msn,1,waveform looks different,10,fi210713a.0693,GO_R,211.0,1043,NaN,...,2194,0.0,"[[283, 305], [439, 454], [1254, 1286]]",None,0,[],fi210713,a,693,fi210713a
4,9026,msn,1,waveform looks different,10,fi210713a.0475,GO_L,189.0,1018,NaN,...,2169,0.0,"[[188, 209], [1207, 1241]]",None,180,[],fi210713,a,475,fi210713a


In [3]:
print(cell_df[['cell_type', 'cell_ID']].drop_duplicates('cell_ID')['cell_type'].value_counts())

cell_df['grade'].value_counts()


cell_type
pu msn     2422
msn        1761
hfdp        466
lfd         155
gpi         140
tan          92
pu tan       87
lfdb         31
fsn          26
unknown      24
fiber        22
bd           10
fiber?        8
fef           6
tan           1
ctx           1
Name: count, dtype: int64


grade
8     1849206
9      686280
7      543571
6       83350
10      23017
Name: count, dtype: int64

In [4]:
cell_df['neural_data'].apply(lambda x: isinstance(x, list) and len(x) == 0).sum() / cell_df.shape[0]

np.float64(0.0)

In [5]:
cell_df[cell_df['neural_data'].apply(lambda x: isinstance(x, list) and len(x) == 0)] #.to_csv('empty_neural_data_cells.csv', index=True, header=True)

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session


In [6]:
cell_df[cell_df['neural_data'].apply(lambda x: x.size == 0)]

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9026,msn,1,waveform looks different,10,fi210713a.0529,STOP_R_SSD2,NaN,921,1005.0,...,1605,0.0,"[[248, 296], [569, 586]]",None,0,[],fi210713,a,529,fi210713a
1,9025,msn,1,NaN,10,fi210713a.0169,STOP_R_SSD4,236.0,921,1125.0,...,1726,0.0,"[[36, 63], [364, 397], [1157, 1190]]",None,0,[],fi210713,a,169,fi210713a
2,9026,msn,1,waveform looks different,10,fi210713a.0457,STOP_L_SSD1,NaN,1093,1117.0,...,1717,0.0,"[[133, 165], [323, 352], [444, 526], [536, 547]]","[338, 444, 447, 449]",180,[],fi210713,a,457,fi210713a
3,9026,msn,1,waveform looks different,10,fi210713a.0693,GO_R,211.0,1043,NaN,...,2194,0.0,"[[283, 305], [439, 454], [1254, 1286]]",None,0,[],fi210713,a,693,fi210713a
4,9026,msn,1,waveform looks different,10,fi210713a.0475,GO_L,189.0,1018,NaN,...,2169,0.0,"[[188, 209], [1207, 1241]]",None,180,[],fi210713,a,475,fi210713a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3185395,3673,msn,33,NaN,7,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,[],fi211110,a,2292,fi211110a
3185405,3694,pu msn,54,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,[],fi211110,a,2292,fi211110a
3185407,3705,pu msn,65,NaN,9,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,[],fi211110,a,2292,fi211110a
3185411,3711,pu msn,71,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,[],fi211110,a,2292,fi211110a


In [7]:
cell_df[cell_df['cell_ID'] == 9867]

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
1802990,9867,msn,2,NaN,9,fi210824a.0582,STOP_L_SSD1,NaN,1046,1130.0,...,1830,0.0,"[[176, 211], [700, 715]]",None,180,"[177.7, 1132.9, 1314.35, 1369.47]",fi210824,a,582,fi210824a
1803044,9867,msn,2,NaN,9,fi210824a.0270,GO_L,331.0,942,NaN,...,2093,0.0,"[[727, 743], [867, 872], [1273, 1305]]",None,180,"[1105.8, 1375.23, 2043.58]",fi210824,a,270,fi210824a
1803054,9867,msn,2,NaN,9,fi210824a.0424,CONT_R_SSD1,349.0,900,984.0,...,2051,0.0,"[[179, 215], [783, 789], [1249, 1280]]",None,0,[238.97],fi210824,a,424,fi210824a
1803078,9867,msn,2,NaN,9,fi210824a.0475,CONT_R_SSD2,383.0,984,1116.0,...,2135,0.0,"[[1367, 1401]]",None,0,"[325.23, 428.53, 697.27, 1553.82, 1855.82]",fi210824,a,475,fi210824a
1803088,9867,msn,2,NaN,9,fi210824a.0399,GO_R,253.0,903,NaN,...,2054,0.0,"[[173, 207], [1156, 1188]]",None,0,"[1169.15, 1372.92, 1435.7]",fi210824,a,399,fi210824a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1821885,9867,msn,2,NaN,9,fi210824a.0367,GO_R,314.0,945,NaN,...,2096,0.0,"[[22, 53], [170, 203], [408, 431], [1259, 1294]]",None,0,"[173.45, 509.98, 1357.57, 1892.35]",fi210824,a,367,fi210824a
1821936,9867,msn,2,NaN,9,fi210824a.0442,CONT_R_SSD4,288.0,954,1182.0,...,2105,0.0,"[[109, 137], [234, 311], [319, 336], [1242, 12...","[124, 234]",0,"[17.5, 1293.42, 1677.07, 1973.22]",fi210824,a,442,fi210824a
1822019,9867,msn,2,NaN,9,fi210824a.0177,GO_R,231.0,1004,NaN,...,2155,0.0,"[[44, 44], [47, 75], [184, 255], [263, 281], [...","[60, 183]",0,"[895.05, 1300.77, 1930.35]",fi210824,a,177,fi210824a
1822044,9867,msn,2,NaN,9,fi210824a.0321,GO_L,234.0,941,NaN,...,2092,0.0,"[[131, 165], [1175, 1210]]",None,180,[2083.75],fi210824,a,321,fi210824a


In [8]:
cell_df['filename'].apply(lambda x: x.split('.')[0]).nunique()

84

In [9]:
df = cell_df[cell_df['neural_data'].apply(lambda x: len(x) == 0)]
df = df[df['grade'] == 8]
df = df[df['session'] == 'fi211109']
df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
64119,3548,msn,3,NaN,8,fi211109a.1988,GO_R,287.0,1026,NaN,...,2177,0.0,"[[92, 141], [901, 916], [1313, 1347]]",None,0,[],fi211109,a,1988,fi211109a
64132,3574,pu msn,29,NaN,8,fi211109a.1988,GO_R,287.0,1026,NaN,...,2177,0.0,"[[92, 141], [901, 916], [1313, 1347]]",None,0,[],fi211109,a,1988,fi211109a
64138,3586,pu msn,41,NaN,8,fi211109a.1988,GO_R,287.0,1026,NaN,...,2177,0.0,"[[92, 141], [901, 916], [1313, 1347]]",None,0,[],fi211109,a,1988,fi211109a
64146,3603,pu msn,58,NaN,8,fi211109a.1988,GO_R,287.0,1026,NaN,...,2177,0.0,"[[92, 141], [901, 916], [1313, 1347]]",None,0,[],fi211109,a,1988,fi211109a
64153,3612,pu msn,67,NaN,8,fi211109a.1988,GO_R,287.0,1026,NaN,...,2177,0.0,"[[92, 141], [901, 916], [1313, 1347]]",None,0,[],fi211109,a,1988,fi211109a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195296,3607,pu msn,62,NaN,8,fi211109a.1236,STOP_R_SSD1,438.0,1053,1101.0,...,1801,0.0,"[[138, 174], [844, 857], [1491, 1505]]",None,0,[],fi211109,a,1236,fi211109a
195303,3615,pu msn,70,NaN,8,fi211109a.1236,STOP_R_SSD1,438.0,1053,1101.0,...,1801,0.0,"[[138, 174], [844, 857], [1491, 1505]]",None,0,[],fi211109,a,1236,fi211109a
195329,1992,msn,9,NaN,8,fi211109a.0232,CONT_L_SSD4,242.0,1029,1257.0,...,2180,0.0,"[[174, 204], [726, 741], [1271, 1303], [1779, ...",None,180,[],fi211109,a,232,fi211109a
195356,2032,pu msn,49,NaN,8,fi211109a.0232,CONT_L_SSD4,242.0,1029,1257.0,...,2180,0.0,"[[174, 204], [726, 741], [1271, 1303], [1779, ...",None,180,[],fi211109,a,232,fi211109a


In [10]:
cell_df[cell_df['cell_ID'] == 2049]['neural_data'].value_counts()

neural_data
[60.81, 370.46, 512.89, 562.79, 616.84, 709.16, 775.86, 915.14, 1007.94, 1074.44, 1130.01, 1488.46, 1629.81, 1689.44, 2086.81, 2184.29]                     1
[38.88, 114.58, 259.56, 404.88, 549.58, 887.41, 902.81, 1008.41, 1097.46, 1216.38, 1307.51, 1419.81, 1520.33, 1876.18, 1922.46, 2079.33]                    1
[214.3, 391.9, 533.22, 760.45, 940.17, 1008.44, 1059.6, 1320.55, 1516.0, 1724.94, 1776.55, 2024.0, 2040.9, 2143.3]                                          1
[118.44, 167.51, 227.81, 300.03, 448.08, 560.79, 756.76, 859.01, 1192.04, 1282.93, 1596.11, 1599.83, 1694.91]                                               1
[90.71, 387.74, 453.23, 515.18, 557.68, 660.73, 1002.44, 1088.89, 1176.76, 1483.71, 1746.98, 1915.26, 2201.04]                                              1
                                                                                                                                                           ..
[79.21, 93.16, 168.61, 493.61, 610.03, 8

In [11]:
cell_df.columns

Index(['cell_ID', 'cell_type', 'maestro_ID', 'problem', 'grade', 'filename',
       'trial_name', 'reaction_time', 'go_cue', 'stop_cue', 'trial_failed',
       'ssd_len', 'ssd_number', 'type', 'first_relevant_saccade',
       'segs_durations', 'segs_times', 'trial_length', 'screen_rotation',
       'saccades', 'blinks', 'dir', 'neural_data', 'session', 'plexon_session',
       'trial_number', 'trial_session'],
      dtype='object')

In [12]:
cell_df = cell_df[cell_df['grade'] <= 8].copy().reset_index(drop=True)
cell_df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,235,msn,1,NaN,7,fi210927a.0621,STOP_L_SSD2,NaN,908,1016.0,...,1716,0.0,"[[179, 210], [355, 445], [460, 473], [704, 728]]","[194, 332]",180,"[233.21, 372.74, 492.74, 725.83, 1434.19]",fi210927,a,621,fi210927a
1,235,msn,1,NaN,7,fi210927a.0578,GO_R,334.0,971,NaN,...,2122,0.0,"[[161, 200], [862, 874], [1305, 1339], [1836, ...",None,0,"[36.43, 621.51, 649.88, 657.31]",fi210927,a,578,fi210927a
2,235,msn,1,NaN,7,fi210927a.0642,STOP_L_SSD3,105.0,1045,1213.0,...,1913,0.0,"[[202, 239], [1150, 1186], [1653, 1670]]",None,180,[],fi210927,a,642,fi210927a
3,234,msn,2,NaN,7,fi210927a.0133,GO_R,151.0,1078,NaN,...,2229,0.0,"[[126, 143], [471, 490], [1229, 1263], [1865, ...",None,0,"[631.53, 632.96, 689.98, 714.21, 1250.43, 2081...",fi210927,a,133,fi210927a
4,235,msn,1,NaN,7,fi210927a.0727,GO_L,181.0,983,NaN,...,2134,0.0,"[[114, 152], [488, 504], [1164, 1198]]",None,180,"[778.16, 879.46, 1709.34, 1818.81, 1824.46, 18...",fi210927,a,727,fi210927a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2476122,3716,pu tan,76,NaN,7,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,"[276.46, 369.21, 431.21, 616.64, 724.51, 909.6...",fi211110,a,2292,fi211110a
2476123,3717,pu msn,77,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,"[655.36, 1214.86, 1822.04]",fi211110,a,2292,fi211110a
2476124,3719,pu msn,79,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,"[106.16, 111.03, 118.89, 282.76, 292.91, 429.3...",fi211110,a,2292,fi211110a
2476125,3725,pu msn,85,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,"[157.21, 194.71, 244.76, 1555.06]",fi211110,a,2292,fi211110a


In [13]:
cell_df['session'].nunique()


81

In [14]:
## Cell 3: Trial Type Distribution and Success Rates

# Create summary statistics for plotting
trial_summary = cell_df.groupby(['type', 'trial_failed']).size().reset_index(name='count')
trial_summary['outcome'] = trial_summary['trial_failed'].map({False: 'Success', True: 'Failed'})

# Calculate success rates by trial type
success_rates = cell_df.groupby('type').agg({
    'trial_failed': ['count', 'sum', 'mean']
}).round(3)
success_rates.columns = ['total_trials', 'failed_trials', 'failure_rate']
success_rates['success_rate'] = (1 - success_rates['failure_rate']) * 100
success_rates['failure_rate'] *= 100
print("Success rates by trial type:")
print(success_rates)

# Create the main visualization
plot1 = trial_summary.hvplot.bar(
    x='type', y='count', by='outcome',
    stacked=True,
    title=f'{monkey.title()} - Trial Distribution by Type and Outcome',
    xlabel='Trial Type',
    ylabel='Number of Trials',
    width=600, height=400,
    color=['#2E8B57', '#CD5C5C'],  # Green for success, red for failed
    legend='top_right'
)

plot1.opts(
    fontsize={'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12},
)
# success_rates

Success rates by trial type:
      total_trials  failed_trials  failure_rate  success_rate
type                                                         
CONT        576064          87876          15.3          84.7
GO         1368447          50567           3.7          96.3
STOP        531616         240383          45.2          54.8


:Bars   [type,outcome]   (count)

In [15]:
## Cell 4: Success Rates by Trial Type (Percentage View)
# Create percentage view of success rates
trial_pct = cell_df.groupby('type').apply(
    lambda x: pd.Series({
        'Success': (1 - x['trial_failed'].mean()) * 100,
        'Failed': x['trial_failed'].mean() * 100
    })
).reset_index()

trial_pct_melted = trial_pct.melt(id_vars='type', var_name='outcome', value_name='percentage')

plot2 = trial_pct_melted.hvplot.bar(
    x='type', y='percentage', by='outcome',
    stacked=True,
    title=f'{monkey.title()} - Success Rate by Trial Type (%)',
    xlabel='Trial Type',
    ylabel='Percentage of Trials',
    width=600, height=400,
    color=['#2E8B57', '#CD5C5C'],
    legend='top',
    ylim=(0, 100)
)

plot2
# trial_pct
# trial_pct_melted

/tmp/ipykernel_166964/925790005.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trial_pct = cell_df.groupby('type').apply(


:Bars   [type,outcome]   (percentage)

In [16]:
from bokeh.palettes import Colorblind

# Histogram of cells per grade
print("=== CELLS PER GRADE ANALYSIS ===")

# Get the grade distribution for all cells
grade_counts = cell_df['grade'].value_counts().sort_index()
print(f"Grade distribution:")
for grade, count in grade_counts.items():
    print(f"  Grade {grade}: {count:,} cells")

print(f"\nTotal cells: {len(cell_df):,}")
print(f"Grade range: {cell_df['grade'].min()} - {cell_df['grade'].max()}")
print(f"Mean grade: {cell_df['grade'].mean():.2f}")
print(f"Median grade: {cell_df['grade'].median():.1f}")

# Create bar plot using hvplot with different colors per bar
grade_counts_df = cell_df.groupby('grade').size().reset_index(name='count')

# Create individual bars with different colors
bars = []
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
n_bars = len(grade_counts_df)
palette_size = min(8, max(3, n_bars))
colors = Colorblind[palette_size]

for i, (grade, count) in enumerate(zip(grade_counts_df['grade'], grade_counts_df['count'])):
    bar = hv.Bars([(grade, count)], kdims='grade', vdims='count').opts(
        color=colors[i], 
        alpha=0.8,
        width=10,
    )
    bars.append(bar)

# Overlay all bars
grade_bar = hv.Overlay(bars).opts(
    title=f'{monkey.title()} - Distribution of Cells per Grade',
    xlabel='Grade',
    ylabel='Number of Cells',
    width=700, height=400,
    fontsize=font_dict,
    xticks=list(grade_counts_df['grade'])
)

grade_bar

=== CELLS PER GRADE ANALYSIS ===
Grade distribution:
  Grade 6: 83,350 cells
  Grade 7: 543,571 cells
  Grade 8: 1,849,206 cells

Total cells: 2,476,127
Grade range: 6 - 8
Mean grade: 7.71
Median grade: 8.0


:Overlay
   .Bars.I   :Bars   [grade]   (count)
   .Bars.II  :Bars   [grade]   (count)
   .Bars.III :Bars   [grade]   (count)

In [17]:
# Violin plot of cell distribution by trial type and outcome
print("=== CELL DISTRIBUTION BY TRIAL TYPE AND OUTCOME ===")

# Count cells per trial for all trials - using correct column names
cells_per_trial = cell_df.groupby(['filename', 'type', 'trial_failed']).size().reset_index(name='cell_count')

# # Add outcome labels
cells_per_trial['outcome'] = cells_per_trial['trial_failed'].map({False: 'Success', True: 'Failed'})
cells_per_trial
print(f"Total trials analyzed: {len(cells_per_trial):,}")

print(f"\nCells per trial statistics by type and outcome:")
for trial_type in cells_per_trial['type'].unique():
    for outcome in ['Success', 'Failed']:
        type_outcome_data = cells_per_trial[
            (cells_per_trial['type'] == trial_type) & 
            (cells_per_trial['outcome'] == outcome)
        ]['cell_count']
        if len(type_outcome_data) > 0:
            print(f"  {trial_type} {outcome}: Mean={type_outcome_data.mean():.1f}, "
                  f"Median={type_outcome_data.median():.1f}, Min={type_outcome_data.min()}, "
                  f"Max={type_outcome_data.max()}, Trials={len(type_outcome_data)}")

# Create rotated violin plot with split by outcome using hv.Violin
violin_plot = hv.Violin(
    cells_per_trial, kdims=['type', 'outcome'], vdims='cell_count'
).opts(
    opts.Violin(
        show_legend=True, height=500, width=800,
        violin_color=hv.dim('outcome').str(),
        legend_position='top_right',
        split='outcome',
        title=f'{monkey.title()} - Distribution of Cells per Trial by Type and Outcome',
        xlabel='Trial Type',
        ylabel='Number of Cells per Trial',
        show_grid=True,
        violin_width=2,
        invert_axes=True,  # Keep normal orientation (vertical violins)
        tools=['hover'],
        fontsize=font_dict
    )
)


violin_plot

=== CELL DISTRIBUTION BY TRIAL TYPE AND OUTCOME ===
Total trials analyzed: 101,308

Cells per trial statistics by type and outcome:
  GO Success: Mean=24.5, Median=19.0, Min=1, Max=75, Trials=53731
  GO Failed: Mean=21.7, Median=17.0, Min=1, Max=72, Trials=2326
  STOP Success: Mean=23.5, Median=18.0, Min=1, Max=75, Trials=12417
  STOP Failed: Mean=25.5, Median=20.0, Min=1, Max=75, Trials=9421
  CONT Success: Mean=24.8, Median=20.0, Min=1, Max=75, Trials=19720
  CONT Failed: Mean=23.8, Median=18.0, Min=1, Max=75, Trials=3693


:Violin   [type,outcome]   (cell_count)

In [18]:
cell_df['ssd_len'].describe()

count    2.476127e+06
mean     3.131714e+02
std      1.635597e+02
min      2.400000e+01
25%      1.680000e+02
50%      4.500000e+02
75%      4.500000e+02
max      5.500000e+02
Name: ssd_len, dtype: float64

In [19]:
# Bar plot of cell count by cell type and trial type
print("=== CELL TYPE DISTRIBUTION BY TRIAL TYPE ===")

# Get the cell type distribution by trial type
cell_type_trial_counts = cell_df.groupby(['type', 'cell_type']).size().reset_index(name='count')

# Calculate percentages within each trial type
trial_totals = cell_df.groupby('type').size()
cell_type_trial_counts['percentage'] = cell_type_trial_counts.apply(
    lambda row: (row['count'] / trial_totals[row['type']]) * 100, axis=1
)

print(f"Cell type distribution by trial type:")
for trial_type in cell_df['type'].unique():
    print(f"\n{trial_type} trials:")
    trial_data = cell_type_trial_counts[cell_type_trial_counts['type'] == trial_type].sort_values('count', ascending=False)
    for _, row in trial_data.iterrows():
        print(f"  {row['cell_type']}: {row['count']:,} cells ({row['percentage']:.1f}%)")

print(f"\nTotal cells: {len(cell_df):,}")
print(f"Trial types: {cell_df['type'].unique()}")
print(f"Unique cell types: {cell_df['cell_type'].nunique()}")

# Create grouped bar plot with proper separation and legend
# Use hvplot with explicit handling for the legend
cell_type_bar = cell_type_trial_counts.hvplot.bar(
    x='cell_type', y='count', by='type',
    title=f'{monkey.title()} - Distribution of Cells by Cell Type and Trial Type',
    xlabel='Cell Type',
    ylabel='Number of Cells',
    width=1000, height=600,
    alpha=0.8,
    rot=90,
    color=['#2E8B57', '#FF8C00', '#4169E1'],  # Green for GO, Orange for CONT, Blue for STOP
    legend='top_right'
)

# Apply additional styling options
cell_type_bar = cell_type_bar.opts(
    fontsize=font_dict,
    show_legend=True,
    legend_position='top_right',
    legend_opts={'click_policy': 'hide'}
)

cell_type_bar

=== CELL TYPE DISTRIBUTION BY TRIAL TYPE ===
Cell type distribution by trial type:

STOP trials:
  pu msn: 258,784 cells (48.7%)
  msn: 157,083 cells (29.5%)
  hfdp: 47,401 cells (8.9%)
  gpi: 16,665 cells (3.1%)
  lfd: 13,790 cells (2.6%)
  pu tan: 13,496 cells (2.5%)
  tan: 11,305 cells (2.1%)
  unknown: 3,271 cells (0.6%)
  lfdb: 2,718 cells (0.5%)
  fsn: 1,982 cells (0.4%)
  fiber: 1,961 cells (0.4%)
  fiber?: 1,112 cells (0.2%)
  bd: 976 cells (0.2%)
  fef: 921 cells (0.2%)
  tan : 119 cells (0.0%)
  ctx: 32 cells (0.0%)

GO trials:
  pu msn: 669,583 cells (48.9%)
  msn: 402,757 cells (29.4%)
  hfdp: 121,183 cells (8.9%)
  gpi: 42,692 cells (3.1%)
  pu tan: 34,956 cells (2.6%)
  lfd: 34,752 cells (2.5%)
  tan: 29,010 cells (2.1%)
  unknown: 7,707 cells (0.6%)
  lfdb: 7,133 cells (0.5%)
  fiber: 5,323 cells (0.4%)
  fsn: 5,280 cells (0.4%)
  fiber?: 2,715 cells (0.2%)
  bd: 2,596 cells (0.2%)
  fef: 2,375 cells (0.2%)
  tan : 309 cells (0.0%)
  ctx: 76 cells (0.0%)

CONT trials:
  

:Bars   [cell_type,type]   (count)

In [20]:
# How many succesful trials per cell type and trial type
print("=== SUCCESSFUL TRIALS PER CELL TYPE AND TRIAL TYPE ===")
# Filter for successful trials only
successful_trials = cell_df[cell_df['trial_failed'] == False]
cell_type_trial_counts = successful_trials.groupby(['type', 'cell_type']).size().reset_index(name='count')

cell_type_trial_counts

# Create grouped bar plot with proper separation and legend
# Use hvplot with explicit handling for the legend
successful_cell_type_bar = cell_type_trial_counts.hvplot.bar(
    x='cell_type', y='count', by='type',
    title=f'{monkey.title()} - Successful Trials by Cell Type and Trial Type',
    xlabel='Cell Type',
    ylabel='Number of Successful Trials',
    width=1000, height=600,
    alpha=0.8,
    rot=90,
    color=['#2E8B57', '#FF8C00', '#4169E1'],  # Green for GO, Orange for CONT, Blue for STOP
    legend='top_right'
)
# Apply additional styling options
successful_cell_type_bar = successful_cell_type_bar.opts(
    fontsize=font_dict,
    show_legend=True,
    legend_position='top_right',
    legend_opts={'click_policy': 'hide'}
)   
successful_cell_type_bar


=== SUCCESSFUL TRIALS PER CELL TYPE AND TRIAL TYPE ===


:Bars   [cell_type,type]   (count)

In [ ]:
msn_df = cell_df[cell_df['cell_type'].isin(['msn', 'pu msn'])].copy()
msn_df.attrs['max_grade'] = 8
msn_df.attrs['description'] = "DataFrame filtered to include only MSN and PU MSN cell types with a minimum grade of 8."
msn_df.attrs['update_stop_trial_failures_by_saccade_amplitude'] = True
msn_df.attrs['monkey'] = monkey
msn_df.attrs.update(cell_df.attrs)

# msn_df.to_pickle(save_path / f'msn_{monkey}_cell_trial_data.pkl')
# msn_df.to_pickle(save_path / f'alt_saccade_params_msn_{monkey}_cell_trial_data.pkl')


In [22]:
msn_df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,235,msn,1,NaN,7,fi210927a.0621,STOP_L_SSD2,NaN,908,1016.0,...,1716,0.0,"[[179, 210], [355, 445], [460, 473], [704, 728]]","[194, 332]",180,"[233.21, 372.74, 492.74, 725.83, 1434.19]",fi210927,a,621,fi210927a
1,235,msn,1,NaN,7,fi210927a.0578,GO_R,334.0,971,NaN,...,2122,0.0,"[[161, 200], [862, 874], [1305, 1339], [1836, ...",None,0,"[36.43, 621.51, 649.88, 657.31]",fi210927,a,578,fi210927a
2,235,msn,1,NaN,7,fi210927a.0642,STOP_L_SSD3,105.0,1045,1213.0,...,1913,0.0,"[[202, 239], [1150, 1186], [1653, 1670]]",None,180,[],fi210927,a,642,fi210927a
3,234,msn,2,NaN,7,fi210927a.0133,GO_R,151.0,1078,NaN,...,2229,0.0,"[[126, 143], [471, 490], [1229, 1263], [1865, ...",None,0,"[631.53, 632.96, 689.98, 714.21, 1250.43, 2081...",fi210927,a,133,fi210927a
4,235,msn,1,NaN,7,fi210927a.0727,GO_L,181.0,983,NaN,...,2134,0.0,"[[114, 152], [488, 504], [1164, 1198]]",None,180,"[778.16, 879.46, 1709.34, 1818.81, 1824.46, 18...",fi210927,a,727,fi210927a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2476121,3715,pu msn,75,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,"[202.44, 473.29, 707.81, 842.96, 1147.48, 1493...",fi211110,a,2292,fi211110a
2476123,3717,pu msn,77,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,"[655.36, 1214.86, 1822.04]",fi211110,a,2292,fi211110a
2476124,3719,pu msn,79,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,"[106.16, 111.03, 118.89, 282.76, 292.91, 429.3...",fi211110,a,2292,fi211110a
2476125,3725,pu msn,85,NaN,8,fi211110a.2292,STOP_L_SSD3,486.0,1068,1236.0,...,1936,0.0,"[[256, 289], [771, 785], [1554, 1565]]",None,180,"[157.21, 194.71, 244.76, 1555.06]",fi211110,a,2292,fi211110a


In [23]:
msn_df['screen_rotation'].value_counts()

screen_rotation
0.0    1939269
Name: count, dtype: int64